# 07 · M1.3 — M1.1 + the drift rule + a fitted Student-t, on a fixed 260-step window

## Executive summary (read this first)

**M1.3** puts together the pieces that held up in notebooks 03–06, with **no tuned setting**:

| piece | M0 | M1.1 | **M1.3** |
|---|---|---|---|
| transform | change | per asset type (`transformations.toml`) | per asset type |
| centre | last + h × mean step (drift) | same | **drift rule**: drift for CPI and NFP; the last value for rates, FX, equity factors, unemployment |
| shock | normal | normal | **Student-t, ν fitted** from the window (rates, FX, equity factors); normal for CPI, NFP, UNRATE |
| window | 300 rows | 300 rows | **260 steps** (about one trading year), for the drift, the sd, the correlation and ν |
| width | sd × √steps | same | same (no widen, no EWMA) |

The Student-t: ν = 4 + 6 ÷ k from the window's excess kurtosis k (kept in 4.2–30; k < 0.2 stays normal); each draw's normal
shock is multiplied by `√((ν − 2) ÷ W)`, `W ~ χ²(ν)`, one per draw shared by the unit's Student-t cells (same variance, fatter
tails). The window is the same for every unit, monthly ones included (260 months).

Built in three steps so each piece's effect is visible:

| step | model |
|---|---|
| a | M1.1 + the drift rule (normal, 300 rows) |
| b | + the Student-t, ν fitted (300 rows) |
| c | the window fixed at 260 = **M1.3** |

**The test:** notebook 01's backtest dates and answers; competition score per unit = `Σ weight × mean(category score) ÷
mean(M0's)`. Below 1 beats M0. The drift rule was decided after looking at this block (notebook 06); nothing else is.

Results: §4 (every model side by side), §5–§6 (per unit, family, asset type), §8 (in short). No realized outcome is read.

## 0 — Setup: notebook 01's backtest dates and answers, M0 and M1.1 scores; the rules

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}


## 1 — The model

`m13_draws(U, origin, steps_window, drift_rule, student_t)`. With `steps_window=None` it uses M0's 300 rows (299 steps);
with a number W it uses the last W steps. Checks: (1) with 300 rows, no drift rule and no Student-t it is M1.1, so it
equals M0 exactly where M1.1 does; (2) one unit worked through at the real as-of.

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def m13_draws(U, origin, steps_window=WINDOW, drift_rule=True, student_t=True):
    by_asset, x, log_mode, last = fit(U, origin, steps_window)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    def centre(a, h):
        anchor = 0.0 if log_mode[a] else last[a]
        use_drift = U["rules"][a]["drift"] if drift_rule else True
        return anchor + (s[h] * mu[ai[a]] if use_drift else 0.0)
    mean = np.array([centre(a, h) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((m0.N_DRAWS, len(cells))) @ f.T
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"]) if student_t else None
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(m0.N_DRAWS, 1))), 1.0)
    sm = mean + dev
    out = np.empty((m0.N_DRAWS, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if s[h] <= 0:
            col = np.full(m0.N_DRAWS, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out

same = [u for u, U in UNITS_DATA.items() if "log_diff" not in [r["transform"] for r in U["rules"].values()]]
d = max(float(np.abs(m13_draws(UNITS_DATA[u], UNITS_DATA[u]["asof"], None, False, False) - m0.draws(UNITS / u)).max()) for u in same)
print(f"300 rows, no drift rule, no Student-t, on the {len(same)} units where M1.1 = M0: max |M1.3 - M0| = {d:.1e}")
assert d == 0

U = UNITS_DATA["t2-F2-higher-for-longer-2023"]; a = U["assets"][0]; h = U["horizons"][0]
by_asset, x, log_mode, last = fit(U, U["asof"], WINDOW)
k_ = float(((x[0] - x[0].mean()) ** 4).mean() / ((x[0] - x[0].mean()) ** 2).mean() ** 2 - 3)
nu_ = fit_nu(x, [0]); sd_ = float(np.sqrt(np.cov(x)))
print(f"\nexample {U['unit']} ({a}, h = {h}): {x.shape[1]} steps, last = {last[a]}")
print(f"  drift rule: {a} is {asset_type(a)} -> drift {U['rules'][a]['drift']} -> centre = {last[a]}")
print(f"  sd of the 260 steps = {sd_:.5f} -> width = {sd_:.5f} x sqrt({h}) = {sd_ * np.sqrt(h):.4f}")
print(f"  excess kurtosis k = {k_:.3f} -> nu = 4 + 6 / {k_:.3f} = {nu_:.2f}")
q = [0.01, 0.05, 0.5, 0.95, 0.99]
display(pd.DataFrame({"M0": np.quantile(m0.draws(UNITS / U["unit"])[:, 0, 0], q), "M1.3": np.quantile(m13_draws(U, U["asof"])[:, 0, 0], q)},
                     index=[f"{int(p * 100)}%" for p in q]).T)

300 rows, no drift rule, no Student-t, on the 61 units where M1.1 = M0: max |M1.3 - M0| = 0.0e+00

example t2-F2-higher-for-longer-2023 (UST_10Y, h = 63): 260 steps, last = 4.35
  drift rule: UST_10Y is rates -> drift False -> centre = 4.35
  sd of the 260 steps = 0.07692 -> width = 0.07692 x sqrt(63) = 0.6106
  excess kurtosis k = 0.778 -> nu = 4 + 6 / 0.778 = 11.71


,1%,5%,50%,95%,99%
M0,3.262,3.703,4.666,5.663,6.255
M1.3,2.899,3.463,4.372,5.476,5.945


## 2 — Score the steps at every backtest date

In [3]:
cats = ["marginal", "joint", "tail"]

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)

STEPS = {"a: M1.1 + drift rule": dict(steps_window=None, drift_rule=True, student_t=False),
         "b: + Student-t (nu fitted)": dict(steps_window=None, drift_rule=True, student_t=True),
         "c: window 260 = M1.3": dict(steps_window=WINDOW, drift_rule=True, student_t=True)}

def score_unit(u):
    U = UNITS_DATA[u]
    return [{"unit": u, "family": U["family"], "origin": o, "model": name, **parts(U["card"], m13_draws(U, o, **kw).reshape(500, -1), y)}
            for o, y in BACKTEST[u] for name, kw in STEPS.items()]

t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    BT = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
M13_SCORES = BT[BT.model == "c: window 260 = M1.3"].drop(columns="model").reset_index(drop=True)
print(f"scored {len(BT) // len(STEPS):,} dates x {len(STEPS)} models in {time.time() - t0:.0f}s")

scored 23,635 dates x 3 models in 4s


## 3 — The competition score per unit

In [4]:
mean0 = M0_SCORES.groupby("unit")[cats].mean()
def weights(u):
    U = UNITS_DATA[u]; return m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
def vs_m0(scores):
    m = scores.groupby("unit")[cats].mean()
    return pd.Series({u: sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, weights(u)) if wk) for u in UNIT_IDS})

## 4 — Every model on the same backtest

M1.2 is notebook 05's (Student-t fitted on a window, M0's drift kept); production is notebook 06's (text off).

In [5]:
_w12 = pd.read_parquet(DATA / "m1_2_window_backtest_scores.parquet")
models = {"M0": M0_SCORES, "M1 (log returns)": pd.read_parquet(DATA / "m1_backtest_scores.parquet"), "M1.1 (per asset type)": M11_SCORES,
          **{f"M1.2 (t fitted, {lab})": _w12[_w12.window == W].drop(columns="window") for W, lab in [(252, "1 year"), (300, "300 rows"), (504, "2 years")]},
          "production (text off)": pd.read_parquet(DATA / "production_backtest_scores.parquet"),
          **{name: BT[BT.model == name].drop(columns="model") for name in STEPS}}
fam = pd.Series({u: UNITS_DATA[u]["family"] for u in UNIT_IDS})
SC = {name: vs_m0(sc) for name, sc in models.items()}
LADDER = pd.DataFrame({name: {**{f: s[fam[fam == f].index].mean() for f in sorted(fam.unique())}, "all": s.mean(),
                              "better than M0": f"{(s < 1).sum()}/{len(s)}"} for name, s in SC.items()}).T
LADDER

,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0
M0,1,1,1,1,1,0/103
M1 (log returns),1.324,1.096,1.034,1.016,1.11,36/103
M1.1 (per asset type),0.9955,0.999,0.9993,0.9968,0.9976,25/103
"M1.2 (t fitted, 1 year)",1.008,1.017,1.004,0.998,1.006,45/103
"M1.2 (t fitted, 300 rows)",0.9911,0.9995,0.9982,0.993,0.9954,62/103
"M1.2 (t fitted, 2 years)",0.97,0.9967,0.9761,1.01,0.9902,64/103
production (text off),0.9288,1.011,0.9274,1.003,0.9726,74/103
a: M1.1 + drift rule,0.8986,1.002,0.9204,0.9816,0.9553,82/103
b: + Student-t (nu fitted),0.8919,0.9978,0.9177,0.9773,0.9509,81/103
c: window 260 = M1.3,0.8951,0.9995,0.9194,0.9763,0.9521,81/103


## 5 — M1.3 per unit

In [6]:
m13, prod, m11 = SC["c: window 260 = M1.3"], SC["production (text off)"], SC["M1.1 (per asset type)"]
mean13 = M13_SCORES.groupby("unit")[cats].mean()
rows = []
for u in UNIT_IDS:
    w = weights(u)
    rows.append({"family": UNITS_DATA[u]["family"], "unit": u, "asset type": TYPE[u],
                 **{f"{n} ÷ M0": (mean13.loc[u, k] / mean0.loc[u, k] if wk else np.nan) for k, n, wk in zip(cats, ["CRPS", "variogram", "tail"], w)},
                 "M1.1 vs M0": m11[u], "M1.3 vs M0": m13[u], "production vs M0": prod[u]})
PER_UNIT = pd.DataFrame(rows).set_index(["family", "unit"])
PER_UNIT.sort_values("M1.3 vs M0")

asset type  CRPS ÷ M0  variogram ÷ M0  tail ÷ M0  M1.1 vs M0  M1.3 vs M0  production vs M0
family unit                                                                                                                            
T2-F1  t2-F1-measured-pace-2004                   rates      0.454           0.834     0.7669           1      0.6306            0.5517
       t2-F1-conundrum-2005                       rates     0.5775           0.927     0.7042           1      0.7077            0.6256
T2-F2  t2-F2-considerable-removal-2004            rates     0.6831             NaN     0.8597           1      0.7335            0.7274
T2-F1  t2-F1-normalization-2017                   rates     0.7227             NaN     0.8735           1      0.7657             0.826
       t2-F1-dkk-peg-2019                            fx     0.7565             NaN     0.7924       1.006      0.7668            0.8197
       t2-F1-ai-mom-2024                  equity_factor     0.7351             NaN     0.9194           1      0.7878              0.77
       t2-F1-eur-range-2017                          fx     0.7016          0.8849      0.909      0.9528      0.7981            0.8291
T2-F3  t2-F3-conundrum-joint-2005                 rates     0.7222          0.9871     0.7403           1      0.8053            0.8375
       t2-F3-taper-steepener-2013                 rates     0.8076          0.7579     0.9134           1      0.8139            0.8278
       t2-F3-dots-vs-oil-2014                     rates     0.8243          0.7308     0.9203           1      0.8155             0.833
T2-F2  t2-F2-taper-begins-2013                    rates     0.8168             NaN     0.8149           1      0.8163            0.8327
T2-F1  t2-F1-qe3-end-2014                         rates     0.7426          0.9352     0.8367           1      0.8192            0.8437
       t2-F1-ust-positioning-2018                 rates     0.7624          0.9093     0.8338           1      0.8208            0.8929
T2-F2  t2-F2-draghi-precommit-2014                   fx     0.7982             NaN     0.9147      0.9997      0.8315            0.8414
       t2-F2-whatever-it-takes-2012                  fx     0.7967             NaN     0.9245      0.9944      0.8332            0.8549
T2-F1  t2-F1-jpy-ycc-flex-2018                       fx     0.7788          0.9982     0.7731       1.034      0.8435            0.9337
T2-F2  t2-F2-eur-parity-2022                         fx     0.8502             NaN     0.8354      0.9959      0.8459            0.8358
       t2-F2-ecb-qe-telegraph-2014                   fx     0.8043             NaN     0.9692           1      0.8514            0.8394
T2-F1  t2-F1-cpi-glidepath-2023                     cpi     0.9057          0.8154     0.7784      0.8517      0.8531             1.434
T2-F2  t2-F2-trade-war-escalation-2019            rates     0.8675             NaN     0.8691           1      0.8679            0.8699
T2-F3  t2-F3-inversion-persistence-2019           rates     0.8893           0.911     0.7678           1      0.8715            0.9043
T2-F4  t2-F4-hike-cycle-2021Q4b                   rates     0.8762             NaN      0.865           1       0.873            0.8289
T2-F3  t2-F3-hiking-year-curve-2018               rates     0.8795          0.8089     0.9553           1      0.8735             0.919
       t2-F3-oil-fx-split-2022                       fx     0.8866           0.853      0.912      0.9991      0.8816            0.8992
       t2-F3-dollar-squeeze-2020                     fx     0.8875          0.8339     0.9552       1.009       0.885            0.8867
T2-F2  t2-F2-taper-testimony-2013                 rates     0.8667             NaN     0.9321           1      0.8854            0.8839
T2-F1  t2-F1-chf-highly-valued-2021                  fx     0.8397          0.9234     0.9515       1.037      0.8872            0.8065
T2-F3  t2-F3-rates-into-factors-2022      equity_factor     0.8412          0.9395     0.9246        

## 6 — Per family and per asset type

In [7]:
def summary(g):
    return {"units": len(g), "M1.1 vs M0": g["M1.1 vs M0"].mean(), "M1.3 vs M0": g["M1.3 vs M0"].mean(),
            "production vs M0": g["production vs M0"].mean(), "M1.3 better than M0": f'{(g["M1.3 vs M0"] < 1).sum()}/{len(g)}',
            "M1.3 better than production": f'{(g["M1.3 vs M0"] < g["production vs M0"]).sum()}/{len(g)}',
            "past 1.20 (M1.3)": int((g["M1.3 vs M0"] > 1.2).sum())}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*PER_UNIT.groupby("family"), ("all", PER_UNIT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in PER_UNIT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY)
print("per asset type:"); display(BY_TYPE)

per family:


,units,M1.1 vs M0,M1.3 vs M0,production vs M0,M1.3 better than M0,M1.3 better than production,past 1.20 (M1.3)
family,,,,,,,
T2-F1,23,0.9955,0.8951,0.9288,20/23,14/23,1
T2-F2,27,0.999,0.9995,1.011,17/27,13/27,2
T2-F3,22,0.9993,0.9194,0.9274,21/22,14/22,0
T2-F4,31,0.9968,0.9763,1.003,23/31,18/31,0
all,103,0.9976,0.9521,0.9726,81/103,59/103,3


per asset type:


,units,M1.1 vs M0,M1.3 vs M0,production vs M0,M1.3 better than M0,M1.3 better than production,past 1.20 (M1.3)
asset type,,,,,,,
cpi,2,0.8727,0.8735,1.308,2/2,2/2,0
equity_factor,16,1,0.9758,0.9935,11/16,10/16,0
fx,35,1.001,0.9685,0.9819,29/35,21/35,1
fx+rates,5,0.9948,0.9465,0.9429,5/5,3/5,0
payrolls,1,1,1,1.624,0/1,1/1,0
rates,43,1,0.9323,0.9285,33/43,21/43,2
unemployment,1,1,0.9875,1.036,1/1,1/1,0


## 7 — Save

In [8]:
M13_SCORES.to_parquet(DATA / "m1_3_backtest_scores.parquet", index=False)
PER_UNIT.to_csv(DATA / "m1_3_vs_m0_per_unit.csv"); FAMILY.to_csv(DATA / "m1_3_vs_m0_per_family.csv")
BY_TYPE.to_csv(DATA / "m1_3_vs_m0_by_asset_type.csv"); LADDER.to_csv(DATA / "all_models_vs_m0.csv")
for f in ("m1_3_backtest_scores.parquet", "m1_3_vs_m0_per_unit.csv", "m1_3_vs_m0_per_family.csv", "m1_3_vs_m0_by_asset_type.csv", "all_models_vs_m0.csv"):
    print(f"data/{f:<30} {(DATA / f).stat().st_size / 1e3:8.1f} kB")

data/m1_3_backtest_scores.parquet      867.1 kB
data/m1_3_vs_m0_per_unit.csv            13.7 kB
data/m1_3_vs_m0_per_family.csv           0.5 kB
data/m1_3_vs_m0_by_asset_type.csv        0.6 kB
data/all_models_vs_m0.csv                1.2 kB


## 8 — In short

In [9]:
for name in ["M0", "M1.1 (per asset type)", "a: M1.1 + drift rule", "b: + Student-t (nu fitted)", "c: window 260 = M1.3", "production (text off)"]:
    print(f"  {name:<28} {LADDER.loc[name, 'all']:.4f}   (better than M0 on {LADDER.loc[name, 'better than M0']})")
a = FAMILY.loc["all"]
print(f"M1.3 beats production on {a['M1.3 better than production']} units; per family: "
      + "; ".join(f"{f} {FAMILY.loc[f, 'M1.3 vs M0']:.4f} (production {FAMILY.loc[f, 'production vs M0']:.4f})" for f in FAMILY.index if f != "all"))

  M0                           1.0000   (better than M0 on 0/103)
  M1.1 (per asset type)        0.9976   (better than M0 on 25/103)
  a: M1.1 + drift rule         0.9553   (better than M0 on 82/103)
  b: + Student-t (nu fitted)   0.9509   (better than M0 on 81/103)
  c: window 260 = M1.3         0.9521   (better than M0 on 81/103)
  production (text off)        0.9726   (better than M0 on 74/103)
M1.3 beats production on 59/103 units; per family: T2-F1 0.8951 (production 0.9288); T2-F2 0.9995 (production 1.0114); T2-F3 0.9194 (production 0.9274); T2-F4 0.9763 (production 1.0033)
